# 1. Imports

In [14]:
from pathlib import Path
import gzip
import pandas as pd
import requests
import arrow

# 2. Paths and URL

In [15]:
astorb_url = "https://ftp.lowell.edu/pub/elgb/astorb.dat.gz"

output_path = Path("../data/processed/astorb_reduced.parquet")
temp_path = Path("../data/processed/astorb.dat.gz")

# 3. Download

In [16]:
response = requests.get(astorb_url)
response.raise_for_status()

temp_path.parent.mkdir(parents=True, exist_ok=True)
temp_path.write_bytes(response.content)

113512085

O `raise_for_status()` é importante: se o servidor retornar, por exemplo, um erro HTTP, o notebook para imediatamente em vez de você acabar tentando interpretar uma página de erro como se fosse o `astorb.dat`.

# 4. Read the Fixed Width Format file

In [17]:
column_names = [
    "number",
    "name",
    "H",
    "G",
    "M",
    "omega",
    "Omega",
    "inc",
    "e",
    "a",
]

column_specs = [
    (0, 6),      # number
    (7, 25),     # name
    (42, 47),    # H
    (48, 53),    # G
    (115, 125),  # M
    (126, 136),  # omega
    (137, 147),  # Omega
    (147, 157),  # inc
    (158, 168),  # e
    (168, 181),  # a
]

Aqui eu faria questão de usar os índices Python correspondentes às posições do arquivo.

O `astorb.dat` é um **fixed-width file**, portanto `read_fwf()` é exatamente a ferramenta apropriada:

In [18]:
with gzip.open(temp_path, mode="rt") as file:
    astorb = pd.read_fwf(
        file,
        colspecs=column_specs,
        names=column_names,
    )

# 5. Convert data types

In [19]:
astorb["number"] = pd.to_numeric(
    astorb["number"],
    errors="coerce",
).astype("Int64")

numeric_columns = [
    "H",
    "G",
    "M",
    "omega",
    "Omega",
    "inc",
    "e",
    "a",
]

astorb[numeric_columns] = astorb[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

Aqui eu prefiro não especificar `float64` antecipadamente no `read_fwf()`. Primeiro leio o arquivo e depois faço a conversão dos campos numéricos explicitamente.

# 6. Select and reorder columns

In [20]:
astorb = astorb[
    [
        "number",
        "name",
        "H",
        "G",
        "a",
        "e",
        "inc",
        "Omega",
        "omega",
        "M",
    ]
]

Não há necessidade de criar `astorb_raw` e `astorb_clean`. Como você está trabalhando em um notebook e não precisa preservar as duas versões, eu simplesmente transformaria o próprio DataFrame.

# 7. Basic validation

In [21]:
astorb.info()

<class 'pandas.DataFrame'>
RangeIndex: 1566820 entries, 0 to 1566819
Data columns (total 10 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   number  895910 non-null   Int64  
 1   name    1566820 non-null  str    
 2   H       1566820 non-null  float64
 3   G       1566820 non-null  float64
 4   a       1566820 non-null  float64
 5   e       1566820 non-null  float64
 6   inc     1566820 non-null  float64
 7   Omega   1566820 non-null  float64
 8   omega   1566820 non-null  float64
 9   M       1566820 non-null  float64
dtypes: Int64(1), float64(8), str(1)
memory usage: 135.0 MB


E, para verificar os valores ausentes:

In [22]:
astorb.isna().sum()

number    670910
name           0
H              0
G              0
a              0
e              0
inc            0
Omega          0
omega          0
M              0
dtype: int64

Também vale uma inspeção rápida:

In [23]:
astorb.head()

,number,name,H,G,a,e,inc,Omega,omega,M
0,1,Ceres,3.34,0.15,2.765922,0.079751,10.587505,80.248993,73.208090,295.937548
1,2,Pallas,4.12,0.15,2.769387,0.230707,34.933695,172.887003,310.985633,275.616444
2,3,Juno,5.19,0.15,2.671013,0.255661,12.986942,169.810194,247.894048,285.308714
3,4,Vesta,3.25,0.15,2.361256,0.090227,7.143801,103.700084,151.440042,108.380989
4,5,Astraea,6.96,0.15,2.576761,0.187587,5.359799,141.448055,359.387522,205.288315


In [24]:
astorb.describe()

,number,H,G,a,e,inc,Omega,omega,M
count,895910.0,1.566820e+06,1.566820e+06,1.566820e+06,1.566820e+06,1.566820e+06,1.566820e+06,1.566820e+06,1.566820e+06
mean,447955.5,1.759826e+01,1.501344e-01,2.963193e+00,1.588113e-01,9.294357e+00,1.704496e+02,1.818317e+02,1.763537e+02
std,258627.084172,1.920720e+00,2.988291e-02,6.364757e+00,9.509876e-02,6.646425e+00,1.028792e+02,1.036870e+02,1.038033e+02
min,1.0,-1.260000e+00,0.000000e+00,4.617635e-01,3.748000e-05,6.032000e-03,2.230000e-04,6.210000e-04,3.750000e-04
25%,223978.25,1.672000e+01,1.500000e-01,2.400069e+00,9.265636e-02,4.361469e+00,8.287999e+01,9.200776e+01,8.650003e+01
50%,447955.5,1.760000e+01,1.500000e-01,2.668859e+00,1.473630e-01,7.870960e+00,1.619062e+02,1.837492e+02,1.727245e+02
75%,671932.75,1.847000e+01,1.500000e-01,3.029280e+00,2.043221e-01,1.267180e+01,2.564764e+02,2.714839e+02,2.660206e+02
max,895910.0,9.999000e+01,6.660000e+00,2.705611e+03,9.975172e-01,1.759751e+02,3.599999e+02,3.599998e+02,3.599997e+02


# 8. Export to Parquet

In [25]:
output_path.parent.mkdir(parents=True, exist_ok=True)

astorb.to_parquet(
    output_path,
    index=False,
)

# 9. Remove temporary file

In [26]:
temp_path.unlink()